# Enable Models as a Service (MaaS)

This notebook guides you through enabling MaaS on your RHOAI cluster as a **unified gateway** for both model inference and MCP tool access.

MaaS provides a managed AI gateway with built-in authentication, rate limiting, and API key management — no manual proxy deployment needed.

**What we'll set up:**
1. Verify MaaS platform components
2. Configure model access via MaaS
3. Register MCP servers with the MaaS gateway
4. Create API keys (used for both models and MCP tools)
5. Verify the unified setup

**Prerequisites:**
- RHOAI with MaaS support installed on your OpenShift cluster
- Models deployed via RHOAI Dashboard or InferenceService
- MCP servers deployed in the `mcp-servers` namespace (Phase 1 completed)

## 1. Verify MaaS Deployment

Check that the MaaS platform components are running.

In [ ]:
%%bash
echo "=== MaaS Gateway ==="
kubectl get gateway -n openshift-ingress maas-default-gateway 2>/dev/null || echo "⚠️  MaaS Gateway not found"

echo ""
echo "=== Auth & Rate Limit Policies ==="
kubectl get authpolicy -A 2>/dev/null
kubectl get ratelimitpolicy -A 2>/dev/null

echo ""
echo "=== MaaS API ==="
APP_NS=$(kubectl get ns redhat-ods-applications --no-headers 2>/dev/null && echo redhat-ods-applications || echo opendatahub)
kubectl get pods -n ${APP_NS} -l app.kubernetes.io/name=maas-api 2>/dev/null || echo "⚠️  MaaS API pods not found"

## 2. Get Gateway Endpoint

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "MaaS Gateway Endpoint: ${HOST}"
echo ""
echo "API Endpoints:"
echo "  Models:   ${HOST}/maas-api/v1/models"
echo "  API Keys: ${HOST}/maas-api/v1/api-keys"
echo ""
echo "Save this for IDE configuration."

## 3. Enable MaaS for Models

### Option A: Through the RHOAI Dashboard (Recommended)

1. Navigate to **RHOAI Dashboard → Model Serving**
2. Deploy a model (or edit an existing one)
3. Check the **MaaS** checkbox to enable MaaS gateway access
4. Check the **Require authentication** checkbox to enforce auth

> **⚠️ Important:** Always check **both** the MaaS and Require authentication checkboxes. If only MaaS is checked without Require authentication, the direct model endpoint remains freely accessible, bypassing MaaS auth and rate limiting.

### Option B: Through CLI

Follow the [MaaS documentation](https://opendatahub-io.github.io/models-as-a-service/latest/install/model-setup/) for CLI-based model setup.

## 4. Create an API Key

### Option A: Through the RHOAI Dashboard

1. Navigate to **AI assets → Endpoints → Models as a Service**
2. Click **View** on your model
3. Click **Generate Token** to create an API key
4. Copy and save the key — it is shown only once

### Option B: Through CLI

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "Creating API key..."
API_KEY_RESPONSE=$(curl -sSk \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -X POST \
  -d '{"name": "lab-key", "description": "Key for code assistant lab", "expiresIn": "30d"}' \
  "${HOST}/maas-api/v1/api-keys")

echo "$API_KEY_RESPONSE" | python3 -m json.tool

API_KEY=$(echo $API_KEY_RESPONSE | python3 -c "import sys,json; print(json.load(sys.stdin).get('key',''))")
echo ""
echo "⚠️  Save this API key — it is shown only once!"
echo "   API Key: ${API_KEY}"

## 5. List Available Models

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "=== Available Models via MaaS ==="
curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" | python3 -m json.tool

## 6. Test Inference via MaaS

Replace `MODEL_NAME` and `MODEL_URL` with values from the model list above.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

# Get the first available model
MODELS_JSON=$(curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json")

MODEL_NAME=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['id'] if d.get('data') else '')")
MODEL_URL=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['url'] if d.get('data') else '')")

if [ -z "$MODEL_NAME" ]; then
    echo "⚠️  No models found via MaaS. Deploy a model with MaaS enabled first."
    exit 0
fi

echo "Testing model: ${MODEL_NAME}"
echo "URL: ${MODEL_URL}"
echo ""

# Use OCP token for quick test (replace with API key for production)
curl -sSk "${MODEL_URL}/v1/chat/completions" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -d "{\"model\": \"${MODEL_NAME}\", \"messages\": [{\"role\": \"user\", \"content\": \"Write a Python hello world in one line.\"}], \"max_tokens\": 50}" | python3 -m json.tool

## 7. Test Authorization Enforcement

Verify that unauthenticated requests are rejected.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

MODELS_JSON=$(curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json")
MODEL_NAME=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['id'] if d.get('data') else '')")
MODEL_URL=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['url'] if d.get('data') else '')")

echo "Sending unauthenticated request (expecting 401)..."
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" \
  -H "Content-Type: application/json" \
  -d "{\"model\": \"${MODEL_NAME}\", \"messages\": [{\"role\": \"user\", \"content\": \"Hello\"}], \"max_tokens\": 10}" \
  "${MODEL_URL}/v1/chat/completions")

if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "✅ Auth enforced — unauthenticated request rejected (HTTP ${HTTP_CODE})"
else
    echo "⚠️  Got HTTP ${HTTP_CODE} — check that 'Require authentication' is enabled"
fi

---
## 8. Register MCP Servers with MaaS Gateway

Expose MCP servers through the MaaS gateway so developers can access tools using the same API key and endpoint.

This creates HTTPRoute resources that route MCP traffic through the gateway with auth policies applied.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
GATEWAY_NS="openshift-ingress"
MCP_NS="mcp-servers"

echo "=== Registering MCP Servers with MaaS Gateway ==="
echo ""

# Get deployed MCP server services
MCP_SERVICES=$(kubectl get svc -n ${MCP_NS} -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null)

if [ -z "$MCP_SERVICES" ]; then
    echo "⚠️  No MCP services found in namespace '${MCP_NS}'."
    echo "   Deploy MCP servers first (Phase 1: 1_mcp_servers/2_deploy_mcp_servers.ipynb)"
    exit 0
fi

echo "Found MCP services:"
echo "$MCP_SERVICES" | while read svc; do
    echo "  - ${svc}"
done
echo ""

# Create HTTPRoutes for each MCP server through the MaaS gateway
for SVC_NAME in $MCP_SERVICES; do
    SHORT_NAME=$(echo $SVC_NAME | sed 's/^mcp-//')
    SVC_PORT=$(kubectl get svc ${SVC_NAME} -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}')

    echo "Creating HTTPRoute for ${SVC_NAME} → /mcp/${SHORT_NAME}/..."

    kubectl apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-route-${SHORT_NAME}
  namespace: ${MCP_NS}
  labels:
    maas.opendatahub.io/managed: "true"
    app.kubernetes.io/part-of: mcp-servers
spec:
  parentRefs:
    - name: maas-default-gateway
      namespace: ${GATEWAY_NS}
      sectionName: maas-https
  hostnames:
    - "maas.${CLUSTER_DOMAIN}"
  rules:
    - matches:
        - path:
            type: PathPrefix
            value: /mcp/${SHORT_NAME}
      backendRefs:
        - name: ${SVC_NAME}
          namespace: ${MCP_NS}
          port: ${SVC_PORT:-3000}
      filters:
        - type: URLRewrite
          urlRewrite:
            path:
              type: ReplacePrefixMatch
              replacePrefixMatch: /
EOF
done

echo ""
echo "✅ MCP servers registered with MaaS gateway"
echo ""
echo "MCP endpoints via MaaS:"
for SVC_NAME in $MCP_SERVICES; do
    SHORT_NAME=$(echo $SVC_NAME | sed 's/^mcp-//')
    echo "  https://maas.${CLUSTER_DOMAIN}/mcp/${SHORT_NAME}/sse"
done

### 8.1 Apply Auth Policy for MCP Servers

Protect MCP routes with the same Kuadrant AuthPolicy used for model endpoints.

In [ ]:
%%bash
MCP_NS="mcp-servers"
GATEWAY_NS="openshift-ingress"
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')

echo "Applying AuthPolicy for MCP server routes..."

kubectl apply -f - <<EOF
apiVersion: kuadrant.io/v1
kind: AuthPolicy
metadata:
  name: mcp-servers-auth
  namespace: ${MCP_NS}
  labels:
    maas.opendatahub.io/managed: "true"
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-route-sequential-thinking
  defaults:
    rules:
      authentication:
        "maas-api-key":
          apiKey:
            selector:
              matchLabels:
                maas.opendatahub.io/managed: "true"
          credentials:
            authorizationHeader:
              prefix: Bearer
EOF

echo ""
echo "✅ AuthPolicy applied — MCP routes require MaaS API key"
echo ""
echo "⚠️  Note: The same API key used for model inference works for MCP access."

### 8.2 Verify MCP Server Registration

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "=== MCP HTTPRoutes via MaaS Gateway ==="
kubectl get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true 2>/dev/null || echo "  No MCP routes found"

echo ""
echo "=== Health Check (via gateway) ==="
for route in $(kubectl get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    SHORT_NAME=$(echo $route | sed 's/^mcp-route-//')
    URL="${HOST}/mcp/${SHORT_NAME}/sse"
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
      -H "Authorization: Bearer $(oc whoami -t)" \
      "${URL}")
    if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
        echo "  ✅ ${SHORT_NAME}: ${URL} (HTTP ${HTTP_CODE})"
    else
        echo "  ⚠️  ${SHORT_NAME}: ${URL} (HTTP ${HTTP_CODE})"
    fi
done

## Summary

MaaS is now enabled as a **unified gateway** for both models and MCP tools:

| Component | Managed By | Status |
|-----------|-----------|--------|
| MaaS Gateway | RHOAI Operator | Running |
| Auth (Authorino) | RHOAI Operator | Running |
| Rate Limiting (Limitador) | RHOAI Operator | Running |
| MaaS API | RHOAI Operator | Running |
| MCP HTTPRoutes | Lab setup | Running |

| Feature | How |
|---------|-----|
| List models | `GET /maas-api/v1/models` |
| Create API key | `POST /maas-api/v1/api-keys` or RHOAI Dashboard |
| Model inference | `POST <model-url>/v1/chat/completions` with API key |
| MCP tool access | `GET /mcp/<server-name>/sse` with same API key |

## Next Steps

→ `3_test_model_serving.ipynb` — Test inference, streaming, and rate limiting
→ `4_test_mcp_servers.ipynb` — Test MCP server access through the gateway
→ `5_ide_configuration.ipynb` — Configure your IDE to use MaaS endpoints